## Grid SANOS Market Generation Test

This notebook computes historic time series for implied vols, fitting error, DLV vs a gixed NMS grid.



In [1]:

import os
os.environ["JAX_ENABLE_X64"] = "True"

%load_ext autoreload
%autoreload 1

%aimport quantdata
%aimport quantdata.biz_calendar
%aimport quantdata.widb
%aimport quantdata.widb.client
%aimport quantdata.widb.day
%aimport quantdata.widb.arbfree
%aimport quantdata.widb.options_filter
%aimport quantdata.widb.timeseries
%aimport quantdata.dysanos
%aimport quantdata.utils.backbone
%aimport quantdata.sanos.sanos_tools
%aimport quantdata.sanos.arbfree
%aimport quantdata.sanos.sanos
%aimport quantdata.sanos.sanos_jax
%aimport quantdata.sanos.sanos_torch_model


In [ ]:

import math

from cdxcore.bs import bs
from cdxcore.dynaplot import figure, colors_tableau
from cdxcore.err import verify_inp
from cdxcore.subdir import SubDir
from cdxcore.uniquehash import unique_hash8, named_unique_filename48_8
from cdxcore.util import fmt_list
from cdxcore.verbose import Context
from cdxcore.version import version
from matplotlib import cm
from matplotlib.colors import Normalize
from scipy.stats import norm
from quantdata.biz_calendar import DAYS_PER_YEAR, NormalizedStrikes, to_date, NMS5, NMS, DateRange
from cdxcore.pretty import PrettyObject
import numpy as np
from dataclasses import dataclass, field    
import warnings
import scipy as scipy
from collections.abc import Callable
import ipynbname
from quantdata.sanos.sanos import SanosFitter
from quantdata.sanos.sanos_jax import fit_dlvsqs

from quantdata.widb.options_filter import MkArbFree, RawDayFilter, ExpiryFilter, OptionsFilter, filter_options_day
from datetime import date
from quantdata.pool import PoolConfig

DEFAULT_EXPIRIES = np.array([1./255., 3./255., 5./255., 10./255., 20./255., 40./255.,0.5,1.], dtype=np.float64)
DEFAULT_EXPIRIES = np.array([2./255., 5./255., 10./255., 40./255.,0.5,1.], dtype=np.float64)
DEFAULT_NUMS     = (-2,1)

_cache = SubDir("!/.cache_"+ipynbname.name(), fmt=SubDir.PICKLE)

@version("0.1")
@dataclass(frozen=True)
class SanosGridDefinition:
    """
    Extracts from real market data a standardized grid of option prices.

    This standardized grid can then be used in subsequent calculations to fit a uniform SANOS model.
    """

    # grid definition
    grid_expiries    : np.array = field( default_factory=lambda: DEFAULT_EXPIRIES )   # expiries, in years
    grid_range       : tuple = DEFAULT_NUMS  # range for the grid
    grid_num_strikes : int = 20              # number of strikes in the grid
    mode             : str = "nms"           # mode for the grid: "nms" or "strikes"#
    nms_eps          : float = NMS.EPS       # 1 hour

    # interpolation
    smoothness       : float = 0.25          # SANOS smoothness for interpolation
    min_dlv          : float = 0.01          # Minimum DLV used for linear programming.
    max_dlv          : float = 2.            # Maximum DLV used for linear programming.

    def __post_init__(self):
        verify_inp( self.mode in ("nms", "strikes"), lambda : f"Expected mode to be 'nms' or 'strikes' but found {self.mode}.")
        verify_inp( self.grid_num_strikes >= 2, lambda : f"Expected grid_num_strikes to be at least 2 but found {self.grid_num_strikes}.")
        verify_inp( self.grid_expiries[0] > 0. and np.all(np.diff(self.grid_expiries) > 0.), lambda : f"Expected grid expiries to be positive and increasing but found {self.grid_expiries}.")
        verify_inp( len(self.grid_range) == 2 and self.grid_range[0] < 0 and self.grid_range[1] > 0, lambda : f"Expected grid_range to be a tuple of two numbers (lo, hi) with lo<0 and hi>0 but found {self.grid_range}.")
        verify_inp( self.smoothness is None or (self.smoothness >= 0. and self.smoothness < 1.), lambda : f"Expected smoothness to be None or a number between 0 and 1 but found {self.smoothness:.3g}.")
        verify_inp( self.min_dlv >= 0. and self.max_dlv > self.min_dlv, lambda : f"Expected min_dlv to be non-negative and max_dlv to be greater than min_dlv but found min_dlv={self.min_dlv:.3g} and max_dlv={self.max_dlv:.3g}." )
    
    def __str__(self):
        expiries = "("
        for exp in self.grid_expiries[0]:
            expiries += f"{exp*DAYS_PER_YEAR:.1f},"
        expiries = expiries[:-1] + ")"
        str_smoothness = f"sth={self.smoothness:.2f}" if self.smoothness is not None else "lin"
        return f"{self.mode},rg={self.grid_range},nm={self.grid_num_strikes},xp={expiries[:40]};{str_smoothness},mid={self.min_dlv:.3g},mxd={self.max_dlv:.3g}"

    @property
    def T(self) -> int:
        return len(self.grid_expiries)
    @property
    def N(self) -> int:
        return self.grid_num_strikes
    @property
    def TN(self) -> tuple:
        return (self.T, self.N)
    
def _uid_int_grid( ticker, d, raw_day_filter, expiry_filter, options_filter, mk_arb_free, grid_definition ):
    return f"{named_unique_filename48_8(f"grid {grid_definition}", grid_definition, raw_day_filter, expiry_filter, options_filter, mk_arb_free )}/{ticker}/{d}"      

@_cache(fmt=SubDir.PYTREE_HDF5).cache(
    "0.13", dependencies=[SanosGridDefinition, MkArbFree, SanosFitter, OptionsFilter, RawDayFilter, ExpiryFilter, MkArbFree, filter_options_day],
    uid=_uid_int_grid
def interpolate_grid( 
                ticker : str, d : date, *,
                options_filter : OptionsFilter = OptionsFilter(),
                raw_day_filter : RawDayFilter = RawDayFilter(), 
                expiry_filter: ExpiryFilter = ExpiryFilter(),
                mk_arb_free : MkArbFree = MkArbFree(),                 
                grid_definition : SanosGridDefinition = SanosGridDefinition(),
                message_context : Callable|str|None = None,
                verbose : Context = Context.all ) -> dict|None:
    """
    Interpolate the market for a given target grid        
    """

    def message(with_cols : bool = True) -> str:
        if message_context is None:
            return ""
            
        if callable(message_context):
            msg = message_context()
        else:
            msg = str(message_context)
        if msg == "":
            return msg
        msg = msg[0].upper() + msg[1:]
        return msg if not with_cols else msg + ": "

    def _generate_grid(expiries, vols):
        """ Generate grid which contains '1' """

        NMS = NormalizedStrikes(grid_definition.nms_eps)

        # generate strikes which contain '1'
        dist = float(grid_definition.grid_range[1] - grid_definition.grid_range[0])
        assert dist > 0., f"Internal error: expected grid_range to have positive distance but found {grid_definition.grid_range}."
        Nlo  = min(max(2, float(grid_definition.grid_num_strikes*abs(grid_definition.grid_range[0]))//dist),grid_definition.grid_num_strikes-3)
        Nhi = grid_definition.grid_num_strikes - Nlo - 1
        assert Nlo>=2 and Nhi>=2, f"{message()}Internal error: error {Nlo} {Nhi} for grid_range {grid_definition.grid_range} and grid_num_strikes {grid_definition.grid_num_strikes}."
        raw_strikes = np.concatenate( ( np.linspace(grid_definition.grid_range[0], 0., int(Nlo), endpoint=False), np.linspace(0., grid_definition.grid_range[1], int(Nhi)+1) ) )
        assert len(raw_strikes)==grid_definition.N, f"{message()}Internal error: generated number of moneyness levels {len(raw_strikes)} does not match expected {grid_definition.N}."

        if grid_definition.mode == "strikes":
            strikes = np.full( (grid_definition.T, grid_definition.N), raw_strikes[None,:] )
            nms     = NMS.to_nms( k=strikes, t=expiries[:,None], vol=vols[:,None], eps_vol = vols[0] ) 
        else:
            nms     = np.full( (grid_definition.T, grid_definition.N), raw_strikes[None,:] )
            strikes = NMS.from_nms( k=nms, t=expiries[:,None], vol=vols[:,None], eps_vol = vols[0] ) 
        assert np.max( np.diff(strikes[:,0]) ) <= 0., f"{message()}Internal error: expected lower strikes to be decreasing but found {np.max( np.diff(strikes[:,0]) )}."
        assert np.min( np.diff(strikes[:,-1]) ) >= 0., f"{message()}Internal error: expected upper strikes to be increasing but found {np.min( np.diff(strikes[:,-1]) )}."

        return nms, strikes

    # load data
    # ---------

    data    = filter_options_day(ticker,d, mk_arb_free=mk_arb_free, options_filter=options_filter, raw_day_filter=raw_day_filter, expiry_filter=expiry_filter, verbose=verbose, raise_if_bad_day=False )
    if data is None:
        return None

    strikes = data['pure_strike'].to_numpy()
    ixs     = np.argsort(strikes)
    strikes = strikes[ixs]
    tte_y   = data['tte_y'].to_numpy()[ixs]
    bid     = data['pure_bid_price'].to_numpy()[ixs]
    ask     = data['pure_ask_price'].to_numpy()[ixs]
    del ixs

    # split into per-expiry
    # ---------------------

    market = PrettyObject()
    market.uid      = unique_hash8( raw_day_filter, expiry_filter, options_filter, mk_arb_free )
    market.expiries = np.unique(tte_y)
    market.strikes  = []
    market.bids     = []
    market.asks     = []

    for expiry in market.expiries:
        mask = tte_y == expiry
        market.strikes.append( strikes[mask] )
        market.bids.append( bid[mask] )
        market.asks.append( ask[mask] )

    assert set(market) >= {'expiries', 'strikes', 'bids', 'asks'}, f"{message()}Expected market to have keys 'expiries', 'strikes', 'bids', 'asks' but found {set(market)}"

    str_smth = f"{grid_definition.smoothness:.2f}-smooth" if grid_definition.smoothness is not None else "linear"

    # fit a smooth SANOS model to the market
    # --------------------------------------

    with verbose.write_t(lambda : f"\r{message()}Fitting {str_smth} SANOS to {sum( len(s) for s in market.strikes )} strikes... ", end="") as tme:
        mk_arb_free  = MkArbFree(smoothness=grid_definition.smoothness, surface_expiry_linear=True, min_dlv=grid_definition.min_dlv, max_dlv=grid_definition.max_dlv )
        af_output    = mk_arb_free( strikes = market.strikes, expiries = market.expiries, bids = market.bids, asks = market.asks, verbose=verbose.quiet )
        fit_result   = af_output.compute_market_fit( expiries=market.expiries, strikes=market.strikes, bids=market.bids, asks=market.asks, and_vols=False )
        verbose.write(f"done; median error is {fit_result.median_err:.2f} half-spreads for the {fit_result.pcnt_fit*100:.1f}% of options which were not fitted within bid-ask spread. This took {tme}.", head=False )

    # NMS grid
    # --------

    nms_grid = PrettyObject()

    with verbose.write_t(lambda : f"\rTraining {str_smth} SANOS on NMS {grid_definition.grid_nms} grid:") as tme:
        if grid_definition.grid_expiries[-1] > market.expiries[-1]:
            warnings.warn(f"{message()}Warning {str_smth} SANOS: the maximum grid expiry {grid_definition.grid_expiries[-1]:.4g} is greater than the maximum market expiry {market.expiries[-1]:.4g}.")

        T                   = len( grid_definition.grid_expiries )
        N                   = grid_definition.grid_num_nms
        nms_grid.expiries   = np.asarray( grid_definition.grid_expiries )
        nms_grid.atm_vols   = af_output.interpolate_atm_vols( nms_grid.expiries )
        nms_grid.nms,\
        strikes             = _generate_grid(nms_grid.expiries, nms_grid.atm_vols)

        nms_grid.mids,\
        nms_grid.mid_vols   = af_output.price( expiries=nms_grid.expiries[:,None], strikes=strikes, and_vols=True, mode="surface" )
        nms_grid.xstrikes   = np.zeros((T,grid_definition.grid_num_strikes+2), dtype=strikes.dtype)
        nms_grid.xstrikes[:,1:-1] = strikes
        nms_grid.xstrikes[:,0]    = strikes[:,0]/2.
        nms_grid.xstrikes[:,-1]   = strikes[:,-1]*1.5
        del strikes

        # fit and price market4
        sanos           = SanosFitter(smoothness=grid_definition.smoothness, min_dlv=grid_definition.min_dlv, max_dlv=grid_definition.max_dlv )
        sn_grid         = sanos(    expiries = nms_grid.expiries, 
                                    strikes  = [ nms_grid.xstrikes[t,1:-1] for t in range(T) ],
                                    mids     = [ nms_grid.mids[t,:] for t in range(T) ], 
                                    xstrikes = [ nms_grid.xstrikes[t,:] for t in range(T) ],
                                    vols     = nms_grid.atm_vols,
                                    message_context = lambda : message(with_cols=False),
                                    verbose=verbose(1))
        nms_grid.fits,\
        nms_grid.fit_vols    = sn_grid( expiries=nms_grid.expiries[:,None], strikes=nms_grid.xstrikes[:,1:-1], and_vols=True )
        nms_grid.dlvsqs      = sn_grid.dlvsqs( min_dlv=0., max_dlv=None, on_all_strikes=False ) # do not use DLV bounds here!
        err_vols             = np.median( np.abs( nms_grid.fit_vols - nms_grid.mid_vols ) )
        verbose.write(f"Trained {str_smth} SANOS on NMS {grid_definition.grid_nms} grid; this took {tme}. This median match to implied vols was {err_vols:.2f}.")

    assert isinstance( nms_grid.fit_vols, np.ndarray ) , f"{message()}Internal error: expected nms_grid.fit_vols to be a numpy array but found {type(nms_grid.fit_vols)}."
    assert isinstance( nms_grid.dlvsqs, list), f"{message()}Internal error: expected nms_grid.dlvsqs to be a list but found {type(nms_grid.dlvsqs)}."
    nms_grid.dlvsqs       = np.stack( nms_grid.dlvsqs, axis=0 )

    # grid
    assert nms_grid.expiries.shape == (T,), f"{message()}Internal error: expected nms_grid.expiries to have shape {(T,)} but found {nms_grid.expiries.shape}."
    assert nms_grid.atm_vols.shape == (T,), f"{message()}Internal error: expected nms_grid.atm_vols to have shape {(T,)} but found {nms_grid.atm_vols.shape}."
    assert nms_grid.strikes.shape == (T,N), f"{message()}Internal error: expected nms_grid.strikes to have shape {(T,N)} but found {nms_grid.strikes.shape}."
    assert nms_grid.xstrikes.shape == (T,N+2), f"{message()}Internal error: expected nms_grid.strikes to have shape {(T,N)} but found {nms_grid.strikes.shape}."
    assert nms_grid.nms.shape == (N,), f"{message()}Internal error: expected nms_grid.nms to have shape {(N,)} but found {nms_grid.nms.shape}."

    # interpolated market
    assert nms_grid.mids.shape == (T,N), f"{message()}Internal error: expected nms_grid.mids to have shape {(T,N)} but found {nms_grid.mids.shape}."
    assert nms_grid.mid_vols.shape == (T,N), f"{message()}Internal error: expected nms_grid.mid_vols to have shape {(T,N)} but found {nms_grid.mid_vols.shape}."

    # fitted with the grid SANOS
    assert nms_grid.fits.shape == (T,N), f"{message()}Internal error: expected nms_grid.fits to have shape {(T,N)} but found {nms_grid.fits.shape}."
    assert nms_grid.fit_vols.shape == (T,N), f"{message()}Internal error: expected nms_grid.fit_vols to have shape {(T,N)} but found {nms_grid.fit_vols.shape}."
    assert nms_grid.dlvsqs.shape == (T,N), f"{message()}Internal error: expected nms_grid.dlvsqs to have shape {(T,N)} but found {nms_grid.dlvsqs.shape}."

    return dict(nms_grid)

def _uid_int_ts( ticker, date_range, raw_day_filter, expiry_filter, options_filter, mk_arb_free, grid_definition ):
    return f"{named_unique_filename48_8(f"ts {grid_definition}", grid_definition, raw_day_filter, expiry_filter, options_filter, mk_arb_free )}/{ticker}/{date_range}"      

@_cache(fmt=SubDir.PYTREE_HDF5).cache(
    "0.15", dependencies=[MkArbFree, SanosFitter, SanosBenchmarkConfig],
    uid=_uid_int_ts)
def interpolate_time_series( 
                 ticker : str, 
                 date_range : DateRange, *,
                 grid_definition : SanosGridDefinition = SanosGridDefinition(),
                 options_filter : OptionsFilter = OptionsFilter(),
                 raw_day_filter : RawDayFilter = RawDayFilter(), 
                 expiry_filter: ExpiryFilter = ExpiryFilter(),
                 mk_arb_free : MkArbFree = MkArbFree(),
                 pool_config : PoolConfig = PoolConfig.off(),
                 verbose: Context = Context.all ):

    pool  = pool_config.pool()
    dates = date_range.get_business_days()

    results = {}
    cnt = 0
    tme = verbose.timer()

    for d, nms_grid in pool.parallel( { d: pool.delayed(interpolate_grid)( 
                                                                    ticker=ticker, d=d,
                                                                     grid_definition=grid_definition, 
                                                                     mk_arb_free=mk_arb_free, options_filter=options_filter, raw_day_filter=raw_day_filter, expiry_filter=expiry_filter,
                                                                     verbose=verbose(1) if pool.is_no_pool else None ) for d in dates } ):
        cnt +=1 
        if nms_grid is None:
            verbose.write(f"\rFitting {ticker} date {cnt}/{len(dates)} {d}: no data after filtering, skipping.")
        else:
            verbose.write(f"\rFitting {ticker} date {cnt}/{len(dates)} {d}: done; so far this took {tme}...", end='\n' if pool.is_no_pool else '' )
            results[d] = PrettyObject(nms_grid)

    # transform into one big array

    dates = sorted(results)
    results = { d: results[d] for d in dates }    

    r = PrettyObject()
    r.dates      = np.array(dates, dtype="datetime64[D]") # (S,)
    r.expiries   = np.stack( [ results[d].expiries for d in dates ], axis=0 ) # (S, T)
    r.mids       = np.stack( [ results[d].mids for d in dates ], axis=0 ) # (S, T, N)
    r.mid_vols   = np.stack( [ results[d].mid_vols for d in dates ], axis=0 ) # (S, T, N)
    r.xstrikes   = np.stack( [ results[d].xstrikes for d in dates ], axis=0 ) # (S, T, N+2)
    r.nms        = np.stack( [ results[d].nms for d in dates ], axis=0 ) # (S, N)
    r.variances  = np.stack( [ results[d].variances for d in dates ], axis=0 ) # (S, T)

    S  = len(dates)
    T, N = benchmark_config.TN

    assert r.expiries.shape == (S, T), f"Internal error: expected expiries to have shape {(S,T)} but found {r.expiries.shape}."
    assert r.mids.shape == (S, T, N), f"Internal error: expected mids to have shape {(S,T,N)} but found {r.mids.shape}."
    assert r.mid_vols.shape == (S, T, N), f"Internal error: expected mid_vols to have shape {(S,T,N)} but found {r.mid_vols.shape}."
    assert r.xstrikes.shape == (S, T, N+2), f"Internal error: expected xstrikes to have shape {(S,T,N+2)} but found {r.xstrikes.shape}."    
    assert r.nms.shape == (S, N), f"Internal error: expected nms to have shape {(S,N)} but found {r.nms.shape}."
    assert r.variances.shape == (S, T), f"Internal error: expected variances to have shape {(S,T)} but found {r.variances.shape}."
    verbose.write(f"\rFitted {ticker} for {S} dates from {dates[0]} to {dates[-1]} on NMS grid; this took {tme}.")

    return dict(r)

pool_config = PoolConfig(num_workers=4, threading=False)
benchmark_config = SanosBenchmarkConfig(smoothness=0.25)
ts = time_series( "SPX", DateRange(start=date(2025, 4, 1), end=date(2025, 7, 1)), benchmark_config=benchmark_config, pool_config=pool_config, verbose=Context.all )
print(f"/done - found {len(ts['dates'])} dates with data on NMS grid.")


TypeError: NMS.from_nms() got an unexpected keyword argument 'eps_vol'

In [ ]:
from quantdata.sanos.sanos_torch_model import SanosCompressorConfig, sanos_preprocessor, SanosCompressor, torch, BaseEnvironment, FeedForward

env  = BaseEnvironment()
data = sanos_preprocessor( smoothness=benchmark_config.smoothness,
                           expiries=ts['expiries'], 
                           mids=ts['mids'], 
                           mid_vols=ts['mid_vols'],
                           xstrikes=ts['xstrikes'], 
                           nms=ts['nms'],
                           variances=ts['variances'],
                           tdevice=env.device )

S, T, N = ts['mids'].shape

for k,v in data.items():
    print(f"{k}: {v.shape if not v is None else '-'} {v.dtype if not v is None else '-'} {v.device if isinstance(v, torch.Tensor) else ''}")

In [ ]:
S, T, N    = ts['mids'].shape
compressor_config = SanosCompressorConfig(
    num_state = 10, 
    state_config = FeedForward.Config(
        activation = "ReLU",
        num_depth = 2,
        num_width = 200,
        use_bias = True,
    )
)
compressor = SanosCompressor( env,
                              S=S,
                              T=T,
                              N=N,
                              config=compressor_config )

error, _ = compressor( data, training=False )
print(error)

n_epochs = 5000

optimizer = torch.optim.AdamW(
    compressor.parameters(),
    lr=1E-3,
    weight_decay=1e-4,
)

tme = Context().timer()

for epoch in range(n_epochs):
    # -------------------------
    # Training
    # -------------------------
    compressor.train()

    train_loss = 0.0
    n_train = 0

    optimizer.zero_grad(set_to_none=True)
    loss = compressor(data, training=True)

    loss.backward()

    # Optional but often useful
    torch.nn.utils.clip_grad_norm_(compressor.parameters(), max_norm=1.0)

    optimizer.step()
    train_loss = loss.detach().item() 

    if epoch in [0,n_epochs-1] or (epoch+1) % 10 == 0:
        print(f"\rEpoch {epoch+1}/{n_epochs}, Loss: {train_loss:.4g}. So far this took {tme}.", end='', flush=True)

final_error, final_data = compressor( data, training=False )
final_calls = final_data['calls'].cpu().numpy()
xdensities = final_data['xdensities'].cpu().numpy()
dlvsqs = final_data['dlvsqs'].cpu().numpy()
print(f"\rFinal error: {final_error:.4g}. This took {tme}.")


In [ ]:
def display(ticker, benchmark_config):
    str_smooth = f"{benchmark_config.smoothness:.2f}-smooth" if benchmark_config.smoothness is not None else "Linear"

    dates   = ts['dates']
    S, T, N = ts['mids'].shape
    state   = compressor.dlv_state.detach().cpu().numpy()

    with figure(f"Compression {ticker} {str_smooth} Fits - state", col_size=4, row_size=6, columns=state.shape[1] ) as fig:
        colors = colors_tableau()
        for istate in range(state.shape[1]):
            ax = fig.add_subplot(f"state {istate}")
            ax.plot(dates, state[:,istate], color=next(colors))
            ax.set_xlabel("Date")
            ax.tick_params(axis="x", labelrotation=90)

    for ix, d in enumerate(dates):
        with figure(f"Compression {ticker} {str_smooth} Fits date #{ix} {d}", col_size=6, row_size=6, columns=None ) as fig:

            final_calls_t = final_calls[ix,:,:]
            expiries_t    = ts['expiries'][ix,:]
            xstrikes_t    = ts['xstrikes'][ix,:,:]
            mids_t        = ts['mids'][ix,:,:]
            nms_t         = ts['nms'][ix,:]
            mid_vols_t    = ts['mid_vols'][ix,:,:]
            strikes_t     = xstrikes_t[:,1:-1]
            final_vols_t  = bs.implied( k=strikes_t, prices=final_calls_t, sqrtT=np.sqrt(expiries_t[:,None]) )


            X, Y = np.meshgrid(nms_t, expiries_t*DAYS_PER_YEAR)
            ax = fig.add_subplot("Interpolated Market Vols", projection="3d")
            ax.plot_surface(X, Y, mid_vols_t, cmap='viridis', edgecolor='none')
            ax.set_ylabel("Days")
            ax.set_xlabel("NMS")

            X, Y = np.meshgrid(nms_t, expiries_t*DAYS_PER_YEAR)
            ax = fig.add_subplot("Fitted Vols", projection="3d")
            ax.plot_surface(X, Y, final_vols_t, cmap='viridis', edgecolor='none')
            ax.set_ylabel("Days")
            ax.set_xlabel("NMS")

            X, Y = np.meshgrid(nms_t, expiries_t*DAYS_PER_YEAR)
            ax = fig.add_subplot("Abs Vols Difference", projection="3d")
            ax.plot_surface(X, Y, np.abs(final_vols_t - mid_vols_t), cmap='viridis', edgecolor='none')
            ax.set_ylabel("Days")
            ax.set_xlabel("NMS")
            ax.set_zlim( 0., 0.05 )

            ax = fig.add_subplot("DLVs", projection="3d")
            x = np.reshape(nms_t[None,:] + strikes_t*0.,(-1,))
            y = np.reshape((expiries_t*DAYS_PER_YEAR)[:,None] + strikes_t*0.,(-1,))
            z = np.reshape(np.sqrt(dlvsqs[ix,:,:]),(-1,))
            dx = (x.max() - x.min())/100.
            dy = (y.max() - y.min())/100.
            p = ax.bar3d(x=x, y=y, z=0.*z, dx=dx, dy=dy, dz=z, cmap="viridis", edgecolor='none' )
            ax.set_xlabel("NMS")
            ax.set_ylabel("Log-Days")
            ax.set_zlabel("DLV")

            ax = fig.add_subplot("Density", projection="3d")
            x = np.reshape(nms_t[None,:] + strikes_t*0.,(-1,))
            y = np.reshape((expiries_t*DAYS_PER_YEAR)[:,None] + strikes_t*0.,(-1,))
            z = np.reshape(xdensities[ix,:,1:-1],(-1,))
            dx = (x.max() - x.min())/100.
            dy = (y.max() - y.min())/100.
            p = ax.bar3d(x=x, y=y, z=0.*z, dx=dx, dy=dy, dz=z, cmap="viridis", edgecolor='none' )
            ax.set_xlabel("NMS")
            ax.set_ylabel("Log-Days")
            ax.set_zlabel("Density")

            ax = fig.add_subplot("Density", projection="3d")
            x = np.reshape(nms_t[None,:] + strikes_t*0.,(-1,))
            y = np.reshape((expiries_t*DAYS_PER_YEAR)[:,None] + strikes_t*0.,(-1,))
            z = np.reshape(xdensities[ix,:,1:-1],(-1,))
            dx = (x.max() - x.min())/100.
            dy = (y.max() - y.min())/100.
            p = ax.bar3d(x=x, y=y, z=0.*z, dx=dx, dy=dy, dz=z, cmap="viridis", edgecolor='none' )
            ax.set_xlabel("NMS")
            ax.set_ylabel("Log-Days")
            ax.set_zlabel("Density")

            ax = fig.add_subplot("State")
            ax.bar( np.arange(state.shape[-1]), state[ix,:], width=0.2 )


display( "SPX", benchmark_config )
